# GovFriction Lab — Independent CPU statistical reference

A separate Python implementation for benchmarking the browser CPU Research Workbench and optional Cloudflare Worker aggregate test.

**Focus:** recorded nonclosure proportions and Wilson confidence intervals, monthly rates, Pearson chi-square, Cramér's V, and Kaplan–Meier time-to-recorded-closure. No GPU needed.

This notebook deliberately does not import GovFriction Lab's JavaScript computation engine. It uses pandas, NumPy and SciPy independently. Unclosed records represent documented service requests, not verified failures to resolve a physical problem.

Upload the **same frozen CSV** used in GovFriction Lab. Preserve its SHA-256 and study protocol. Use the outputs only after auditing the cohort.

In [ ]:
import hashlib, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
FIELDS={"unique_key","created_date","closed_date","agency","complaint_type","borough"}
SOURCE=None
candidates=sorted(Path("/kaggle/input").rglob("*.csv")) if Path("/kaggle/input").exists() else sorted(Path("data").glob("*.csv"))
if SOURCE is None:
    matches=[p for p in candidates if FIELDS.issubset(set(pd.read_csv(p,nrows=0).columns))]
    if len(matches)!=1:raise RuntimeError(f"Set SOURCE explicitly. Matching candidate files: {matches}")
    SOURCE=matches[0]
SOURCE=Path(SOURCE)
sha256=hashlib.sha256(SOURCE.read_bytes()).hexdigest()
raw=pd.read_csv(SOURCE,dtype=str,keep_default_na=False,encoding="utf-8-sig")
assert FIELDS.issubset(raw.columns)
print("Input:",SOURCE,"SHA-256",sha256,"Rows",len(raw))

In [ ]:
START=pd.Timestamp("2025-01-01")
END=pd.Timestamp("2025-04-01")
SNAPSHOT=pd.Timestamp("2026-11-19")
HORIZON=30
VALID_SERVICES={"Graffiti","Missed Collection (All Materials)"}
VALID_BOROUGHS={"BRONX","BROOKLYN","MANHATTAN","QUEENS","STATEN ISLAND"}
audit={k:0 for k in ["input","invalidCreated","duplicate","scopeExcluded","invalidLocationService","invalidClosed","immature","eligible"]}
audit["input"]=len(raw)
def parsed(text):
    s=str(text).strip()
    if not s:return None
    try:
        t=pd.Timestamp(s)
        return t if not pd.isna(t) and t.tzinfo is None else None
    except (TypeError,ValueError):return None
seen=set();valid=[]
for record in raw.to_dict("records"):
    key=record["unique_key"].strip()
    created=parsed(record["created_date"])
    if not key or created is None:audit["invalidCreated"]+=1;continue
    if key in seen:audit["duplicate"]+=1;continue
    seen.add(key)
    agency=record["agency"].strip().upper()
    service=record["complaint_type"].strip()
    borough=record["borough"].strip().upper()
    if not (START<=created<END and agency=="DSNY" and service in VALID_SERVICES):
        audit["scopeExcluded"]+=1;continue
    if borough not in VALID_BOROUGHS:audit["invalidLocationService"]+=1;continue
    closed_raw=record["closed_date"].strip()
    closed=parsed(closed_raw)
    if (closed_raw and closed is None) or (closed is not None and closed<created):
        audit["invalidClosed"]+=1;continue
    if created>SNAPSHOT:audit["scopeExcluded"]+=1;continue
    if SNAPSHOT-created<pd.Timedelta(days=HORIZON):audit["immature"]+=1;continue
    elapsed=(closed-created).total_seconds()/86400 if closed is not None and closed<=SNAPSHOT else math.inf
    event=int(elapsed<=HORIZON)
    valid.append(dict(service=service,borough=borough,created=created,month=created.strftime("%Y-%m"),
                      nonclosed=1-event,closed=int(event),duration=min(elapsed,HORIZON),event=event))
audit["eligible"]=len(valid)
cases=pd.DataFrame(valid)
if cases.empty:raise RuntimeError("No eligible records.")
print("Audit",audit)

In [ ]:
z=1.959963984540054
def wilson(events,total):
    p=events/total
    den=1+z*z/total
    mid=(p+z*z/(2*total))/den
    half=z*np.sqrt(p*(1-p)/total+z*z/(4*total*total))/den
    return float(max(0,mid-half)),float(min(1,mid+half))
group_rows=[]
for (service,borough),g in cases.groupby(["service","borough"],sort=True):
    n=len(g);events=int(g.nonclosed.sum());low,high=wilson(events,n)
    group_rows.append(dict(service=service,borough=borough,n=n,closed=n-events,
                           nonclosed=events,rate=events/n,wilson95Low=low,wilson95High=high,
                           source_sha256=sha256))
groups=pd.DataFrame(group_rows)
display(groups)
groups.to_csv("advanced_cpu_kaggle_groups.csv",index=False)
monthly=[]
for (service,month),g in cases.groupby(["service","month"],sort=True):
    n=len(g);events=int(g.nonclosed.sum());low,high=wilson(events,n)
    monthly.append(dict(service=service,month=month,n=n,nonclosed=events,rate=events/n,
                        wilson95Low=low,wilson95High=high,source_sha256=sha256))
trends=pd.DataFrame(monthly)
trends.to_csv("advanced_cpu_kaggle_monthly.csv",index=False)
tests=[]
for service in sorted(cases.service.unique()):
    g=groups[groups.service==service]
    table=g[["nonclosed","closed"]].to_numpy()
    if table.shape[0]<2 or np.any(table.sum(axis=0)==0):continue
    chi2,p,dof,expected=chi2_contingency(table,correction=False)
    small=(expected<5).sum()/expected.size>0.2 or (expected<1).any()
    tests.append(dict(service=service,chiSquare=chi2,degreesOfFreedom=dof,
                      pValue=None if small else p,cramersV=np.sqrt(chi2/table.sum()),
                      assumptionsPass=not small,source_sha256=sha256))
chi_results=pd.DataFrame(tests)
display(chi_results)
chi_results.to_csv("advanced_cpu_kaggle_association.csv",index=False)

In [ ]:
# Independent Kaplan–Meier product-limit by service: event = observed closure by horizon;
# longer closures and absent closures are right-censored at horizon.
km_rows=[]
for service,g in cases.groupby("service",sort=True):
    sorted_g=g.sort_values("duration")
    at_risk=len(sorted_g);survival=1.0;greenwood=0.0
    points=[(0.0,1.0,1.0,1.0)]
    for t,part in sorted_g.groupby("duration",sort=True):
        d=int(part.event.sum());c=len(part)-d
        if d:
            survival*=1-d/at_risk
            if at_risk>d:greenwood+=d/(at_risk*(at_risk-d))
            se=survival*np.sqrt(greenwood)
            points.append((float(t),float(survival),
                           max(0.0,float(survival-z*se)),min(1.0,float(survival+z*se))))
        at_risk-=d+c
    for day in sorted(set([0,7,14,HORIZON])):
        if day>HORIZON:continue
        eligible_points=[p for p in points if p[0]<=day]
        time,remaining,low,high=eligible_points[-1]
        km_rows.append(dict(service=service,day=day,remaining=remaining,
                            lower95=low,upper95=high,source_sha256=sha256))
km_table=pd.DataFrame(km_rows)
display(km_table)
km_table.to_csv("advanced_cpu_kaggle_survival.csv",index=False)
manifest=dict(dataset_sha256=sha256,audit=audit,horizonDays=HORIZON,source=str(SOURCE),
              method="independent pandas scipy Wilson/chi-square and Kaplan-Meier calculations",
              outputs=["advanced_cpu_kaggle_groups.csv","advanced_cpu_kaggle_monthly.csv",
                       "advanced_cpu_kaggle_association.csv","advanced_cpu_kaggle_survival.csv"])
Path("advanced_cpu_kaggle_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Wrote advanced CPU reference CSV files and manifest. GPU is not needed.")
